# 🎓 Proyecto: Detección Inteligente de Fraude en Comprobantes Nequi (Arquitectura Dual-Branch)
## Asignatura: Inteligencia Artificial Avanzada | Metodología: Aprendizaje Basado en Retos (ABR)
---
### 📌 1. Identificación y Justificación del Problema
En la detección de fraude sobre billeteras digitales como Nequi existen **dos amenazas fundamentales**:
1. **Aplicaciones Falsas / Clonadas ("Nequi Fake"):** Crean comprobantes sintéticos con diseños apócrifos (cabeceras moradas sin QR, fuentes erróneas, referencias inválidas).
2. **Manipulación Digital de Píxeles (Photoshop / Canva):** Toman un comprobante legítimo y sobreescriben el monto (`¿Cuánto?`) o la fecha.

**Solución de IA de Vanguardia: Red Neuronal Dual-Branch (Siamesa / Fusión Multimodal)**
En lugar de depender de reglas rígidas de color o coordenadas fijas, implementamos una **Red Convolucional Dual** que analiza en paralelo:
* **Rama Visual (RGB):** Aprende a reconocer la presencia del **código QR oficial con marco verde menta (`#84E4BD`)**, la diagramación tipo tiquete y los sellos oficiales.
* **Rama Forense (ELA):** Analiza la homogeneidad de la compresión JPEG para detectar modificaciones y parches en el monto.

In [ ]:
# ====================================================================
# 0. CONFIGURACIÓN DEL ENTORNO Y LIBRERÍAS
# ====================================================================
import os
import random
import glob
from io import BytesIO
from datetime import datetime, timedelta

import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageChops, ImageEnhance
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Fijar semillas aleatorias para reproducibilidad científica
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Entorno configurado con éxito. Aceleración: {device}")

--- 
### 🧪 2. Generador Multimodal de Datos (Comprobantes Oficiales con QR, Apps Falsas y Ediciones)

In [ ]:
COLOR_MINT_QR = (132, 228, 189)     # Verde menta (#84E4BD)
COLOR_PURPLE = (32, 4, 34)          # Morado Nequi (#200422)
COLOR_TEXT_DARK = (20, 20, 25)
COLOR_LABEL_GRAY = (110, 110, 120)
COLOR_DOODLE = (235, 235, 240)

NOMBRES = ["Erick Guardo", "Carlos Rodríguez", "María Gómez", "Andrés Martínez", "Valentina López", "Juan David García"]
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]

def dibujar_qr_nequi(draw, x, y, size=180):
    pad = 12
    draw.rounded_rectangle([(x - pad, y - pad), (x + size + pad, y + size + pad)], radius=8, fill=COLOR_MINT_QR)
    draw.rounded_rectangle([(x, y), (x + size, y + size)], radius=4, fill=(255, 255, 255))
    
    grid_n = 21
    cell_size = size / grid_n
    np.random.seed(x + y)
    for r in range(grid_n):
        for c in range(grid_n):
            es_esq = (r < 7 and c < 7) or (r < 7 and c >= grid_n - 7) or (r >= grid_n - 7 and c < 7)
            es_cntr = (7 <= r <= 13 and 7 <= c <= 13)
            if es_esq:
                if (r in [0, 6] and 0 <= c <= 6) or (c in [0, 6] and 0 <= r <= 6) or (2 <= r <= 4 and 2 <= c <= 4):
                    draw.rectangle([(x + c*cell_size, y + r*cell_size), (x + (c+1)*cell_size, y + (r+1)*cell_size)], fill=COLOR_PURPLE)
                elif (r in [0, 6] and grid_n - 7 <= c < grid_n) or (c in [grid_n - 7, grid_n - 1] and 0 <= r <= 6) or (2 <= r <= 4 and grid_n - 5 <= c <= grid_n - 3):
                    draw.rectangle([(x + c*cell_size, y + r*cell_size), (x + (c+1)*cell_size, y + (r+1)*cell_size)], fill=COLOR_PURPLE)
                elif (r in [grid_n - 7, grid_n - 1] and 0 <= c <= 6) or (c in [0, 6] and grid_n - 7 <= r < grid_n) or (grid_n - 5 <= r <= grid_n - 3 and 2 <= c <= 4):
                    draw.rectangle([(x + c*cell_size, y + r*cell_size), (x + (c+1)*cell_size, y + (r+1)*cell_size)], fill=COLOR_PURPLE)
            elif not es_cntr and np.random.rand() > 0.45:
                draw.rectangle([(x + c*cell_size, y + r*cell_size), (x + (c+1)*cell_size, y + (r+1)*cell_size)], fill=COLOR_PURPLE)
                
    c_x, c_y = x + size/2, y + size/2
    draw.rounded_rectangle([(c_x - 22, c_y - 22), (c_x + 22, c_y + 22)], radius=6, fill=(255, 255, 255))
    draw.text((c_x - 10, c_y - 10), "·N", fill=COLOR_PURPLE)

def crear_comprobante_nequi_actual(datos, ancho=480, alto=880):
    img = Image.new("RGB", (ancho, alto), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    for x_p in range(15, ancho - 15, 8):
        draw.rectangle([(x_p, 25), (x_p + 4, 27)], fill=(200, 200, 210))
        draw.rectangle([(x_p, alto - 25), (x_p + 4, alto - 23)], fill=(200, 200, 210))
    for i in range(150, alto - 50, 45):
        draw.line([(30, i), (ancho - 30, i)], fill=COLOR_DOODLE, width=1)
        
    qr_x = (ancho - 190) // 2
    dibujar_qr_nequi(draw, qr_x, 65, size=190)
    
    info_y = 310
    draw.ellipse([(qr_x - 15, info_y - 2), (qr_x + 10, info_y + 23)], outline=COLOR_TEXT_DARK, width=2)
    draw.text((qr_x - 4, info_y + 2), "i", fill=COLOR_TEXT_DARK)
    draw.text((qr_x + 18, info_y - 4), "¡Escanea este QR con Nequi para", fill=COLOR_TEXT_DARK)
    draw.text((qr_x + 18, info_y + 14), "verificar tu envío al instante!", fill=COLOR_TEXT_DARK)
    
    y_cur = 390
    margen = 55
    draw.text((margen, y_cur), "Para", fill=COLOR_LABEL_GRAY)
    draw.text((margen, y_cur + 24), datos["nombre"], fill=COLOR_TEXT_DARK)
    y_cur += 70
    draw.text((margen, y_cur), "¿Cuánto?", fill=COLOR_LABEL_GRAY)
    draw.text((margen, y_cur + 24), datos["monto"], fill=COLOR_TEXT_DARK)
    y_cur += 75
    draw.text((margen, y_cur), "Número Nequi", fill=COLOR_LABEL_GRAY)
    draw.text((margen, y_cur + 24), datos["telefono"], fill=COLOR_TEXT_DARK)
    y_cur += 70
    draw.text((margen, y_cur), "Fecha", fill=COLOR_LABEL_GRAY)
    draw.text((margen, y_cur + 24), datos["fecha"], fill=COLOR_TEXT_DARK)
    y_cur += 70
    draw.text((margen, y_cur), "Referencia", fill=COLOR_LABEL_GRAY)
    draw.text((margen, y_cur + 24), datos["referencia"], fill=COLOR_TEXT_DARK)
    return img

def crear_comprobante_app_falsa(datos, ancho=480, alto=880):
    """Genera comprobante falso típico con cabecera morada sólida (Nequi Fake)."""
    img = Image.new("RGB", (ancho, alto), color=(248, 248, 252))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0, 0), (ancho, 200)], fill=(98, 42, 115))
    draw.text((30, 40), "NEQUI", fill=(255, 255, 255))
    draw.text((30, 80), "Transferencia exitosa", fill=(230, 230, 240))
    draw.text((30, 110), "Comprobante de pago", fill=(200, 200, 215))
    
    draw.text((30, 240), "Monto enviado", fill=(120, 120, 130))
    draw.text((30, 270), datos["monto"], fill=(20, 20, 20))
    draw.text((30, 350), "Fecha", fill=(120, 120, 130))
    draw.text((30, 380), "15/08/2024", fill=(20, 20, 20))
    draw.text((30, 440), "Referencia", fill=(120, 120, 130))
    draw.text((30, 470), "REF-847291", fill=(20, 20, 20))
    draw.text((30, 530), "Destinatario", fill=(120, 120, 130))
    draw.text((30, 560), datos["nombre"], fill=(20, 20, 20))
    return img

def simular_datos():
    nom = random.choice(NOMBRES)
    tel = f"300 {random.randint(100, 999)} {random.randint(1000, 9999)}"
    val = random.choice([20000, 50000, 100000, 150000, 200000, 350000, 500000, 1000000])
    monto_str = f"$ {val:,.2f}".replace(",", "@").replace(".", ",").replace("@", ".")
    f_base = datetime.now() - timedelta(days=random.randint(0, 30), minutes=random.randint(1, 1440))
    hora_12 = f_base.strftime("%I:%M")
    ampm = "a. m." if f_base.hour < 12 else "p. m."
    fecha = f"{f_base.day:02d} de {MESES[f_base.month - 1]} de {f_base.year} a las {hora_12} {ampm}"
    ref = f"M{random.randint(10000000, 99999999)}"
    return {"nombre": nom, "telefono": tel, "monto": monto_str, "monto_num": val, "fecha": fecha, "referencia": ref}

def generar_muestra(tipo="legitimo"):
    d = simular_datos()
    if tipo == "legitimo":
        img = crear_comprobante_nequi_actual(d)
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=random.choice([60, 75, 85, 92]))
        buf.seek(0)
        return Image.open(buf)
    elif tipo == "app_falsa":
        img = crear_comprobante_app_falsa(d)
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=random.choice([70, 85, 92]))
        buf.seek(0)
        return Image.open(buf)
    else:
        base = crear_comprobante_nequi_actual(d)
        buf = BytesIO()
        base.save(buf, format="JPEG", quality=90)
        buf.seek(0)
        img_edit = Image.open(buf).convert("RGB")
        draw = ImageDraw.Draw(img_edit)
        draw.rectangle([(50, 480), (380, 525)], fill=(245, 246, 248))
        monto_falso = f"$ {d['monto_num']*5:,.2f}".replace(",", "@").replace(".", ",").replace("@", ".")
        draw.text((54, 484), monto_falso, fill=(10, 10, 15))
        buf2 = BytesIO()
        img_edit.save(buf2, format="JPEG", quality=65)
        buf2.seek(0)
        return Image.open(buf2)

# Construir Dataset
for split, n_total in [("train", 400), ("val", 80), ("test", 80)]:
    for cls in ["legitimo", "fraude"]:
        folder = f"dataset_nequi_nuevo/{split}/{cls}"
        os.makedirs(folder, exist_ok=True)
        for i in range(n_total // 2):
            if cls == "legitimo":
                img = generar_muestra("legitimo")
            else:
                tipo_f = random.choice(["app_falsa", "edicion_monto"])
                img = generar_muestra(tipo_f)
            img.save(f"{folder}/{cls}_{i+1:04d}.jpg", quality=85)

print("✓ Dataset balanceado con muestras de App Falsa y Edición Digital generado exitosamente.")

--- 
### 🔬 3. Módulo Forense: Error Level Analysis (ELA)

In [ ]:
def calcular_ela(img_pil, calidad=90, escala=15):
    img_rgb = img_pil.convert("RGB")
    buf = BytesIO()
    img_rgb.save(buf, format="JPEG", quality=calidad)
    buf.seek(0)
    recomprimida = Image.open(buf)
    
    dif = ImageChops.difference(img_rgb, recomprimida)
    extremos = dif.getextrema()
    max_dif = max([ex[1] for ex in extremos]) or 1
    factor = escala * (255.0 / max_dif)
    return ImageEnhance.Brightness(dif).enhance(factor)

# Visualización de Muestras
img_leg = generar_muestra("legitimo")
ela_leg = calcular_ela(img_leg)

img_fake_app = generar_muestra("app_falsa")
img_edit = generar_muestra("edicion_monto")
ela_edit = calcular_ela(img_edit)

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(img_leg); axs[0].set_title("Comprobante Oficial Nequi (QR)"); axs[0].axis("off")
axs[1].imshow(img_fake_app); axs[1].set_title("App Falsa (Cabecera Morada)"); axs[1].axis("off")
axs[2].imshow(ela_edit); axs[2].set_title("Forense ELA (Edición de Monto)"); axs[2].axis("off")
plt.tight_layout()
plt.show()

--- 
### 🧠 4. Arquitectura de Red Neuronal Dual-Branch (Visual RGB + Forense ELA)

In [ ]:
class NequiDualDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.samples = []
        self.transform = transform
        for f in glob.glob(f"{root_dir}/{split}/legitimo/*.jpg"):
            self.samples.append((f, 0.0))
        for f in glob.glob(f"{root_dir}/{split}/fraude/*.jpg"):
            self.samples.append((f, 1.0))
            
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img_rgb = Image.open(path).convert("RGB")
        img_ela = calcular_ela(img_rgb)
        
        if self.transform:
            x_rgb = self.transform(img_rgb)
            x_ela = self.transform(img_ela)
        else:
            t = transforms.ToTensor()
            x_rgb, x_ela = t(img_rgb), t(img_ela)
            
        return x_rgb, x_ela, torch.tensor(label, dtype=torch.float32)

transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(NequiDualDataset("dataset_nequi_nuevo", "train", transform_pipeline), batch_size=16, shuffle=True)
val_loader = DataLoader(NequiDualDataset("dataset_nequi_nuevo", "val", transform_pipeline), batch_size=16, shuffle=False)
test_loader = DataLoader(NequiDualDataset("dataset_nequi_nuevo", "test", transform_pipeline), batch_size=16, shuffle=False)

# Definición del Modelo Dual-Branch
class NequiDualBranchCNN(nn.Module):
    def __init__(self):
        super(NequiDualBranchCNN, self).__init__()
        weights = models.MobileNet_V3_Small_Weights.DEFAULT
        
        # Rama 1: Visual RGB (Estructura, QR, Colores)
        base_rgb = models.mobilenet_v3_small(weights=weights)
        self.branch_rgb = base_rgb.features
        self.pool_rgb = nn.AdaptiveAvgPool2d((1, 1))
        
        # Rama 2: Forense ELA (Artefactos y parches de compresión)
        base_ela = models.mobilenet_v3_small(weights=weights)
        self.branch_ela = base_ela.features
        self.pool_ela = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fusión de ambas ramas (576 + 576 = 1152 dimensiones)
        self.classifier = nn.Sequential(
            nn.Linear(576 * 2, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1)
        )
        
    def forward(self, x_rgb, x_ela):
        f_rgb = self.pool_rgb(self.branch_rgb(x_rgb)).flatten(1)
        f_ela = self.pool_ela(self.branch_ela(x_ela)).flatten(1)
        fused = torch.cat([f_rgb, f_ela], dim=1)
        return self.classifier(fused)

modelo_dual = NequiDualBranchCNN().to(device)
print("✓ Arquitectura Dual-Branch (RGB + ELA) compilada exitosamente.")

--- 
### 🚀 5. Entrenamiento del Modelo Dual

In [ ]:
criterio = nn.BCEWithLogitsLoss()
optimizador = torch.optim.AdamW(modelo_dual.parameters(), lr=0.001, weight_decay=1e-4)
epochs = 8

for epoch in range(epochs):
    modelo_dual.train()
    t_loss, t_corr, total = 0.0, 0, 0
    for x_rgb, x_ela, labels in train_loader:
        x_rgb, x_ela, labels = x_rgb.to(device), x_ela.to(device), labels.to(device).unsqueeze(1)
        optimizador.zero_grad()
        outs = modelo_dual(x_rgb, x_ela)
        loss = criterio(outs, labels)
        loss.backward(); optimizador.step()
        t_loss += loss.item() * x_rgb.size(0)
        preds = (torch.sigmoid(outs) >= 0.5).float()
        t_corr += (preds == labels).sum().item()
        total += labels.size(0)
    print(f"Época [{epoch+1:02d}/{epochs:02d}] - Loss: {t_loss/total:.4f} - Accuracy: {t_corr/total*100:.1f}%")

modelo_dual.eval()
torch.save(modelo_dual.state_dict(), "mejor_modelo_dual_nequi.pth")
print("✓ Modelo Dual-Branch entrenado y fijado en modo evaluación (eval).")

--- 
### 📲 6. Módulo de Prueba Interactiva (Sube tu Comprobante Real)

In [ ]:
from google.colab import files

modelo_dual.eval()
print("📤 Haz clic en 'Elegir archivos' para subir cualquier comprobante:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n" + "="*60)
    print(f"🔍 Analizando comprobante: {filename}")
    print("="*60)
    
    img_pil = Image.open(filename).convert("RGB")
    img_ela = calcular_ela(img_pil)
    
    # Preprocesamiento para ambas ramas
    x_rgb = transform_pipeline(img_pil).unsqueeze(0).to(device)
    x_ela = transform_pipeline(img_ela).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = modelo_dual(x_rgb, x_ela)
        prob_fraude = torch.sigmoid(logits).item()
        
    es_fraude = prob_fraude >= 0.5
    confianza = prob_fraude if es_fraude else (1.0 - prob_fraude)
    
    # Mostrar Diagnóstico Visual Lado a Lado
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img_pil)
    plt.title("Comprobante Evaluado"); plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.imshow(img_ela)
    color_t = "red" if es_fraude else "green"
    titulo = f"🚨 DICTAMEN: FRAUDE ({confianza*100:.1f}%)" if es_fraude else f"✅ DICTAMEN: AUTÉNTICO ({confianza*100:.1f}%)"
    plt.title(titulo, color=color_t, fontweight="bold"); plt.axis("off")
    plt.tight_layout()
    plt.show()
    
    print("\n📋 RESULTADO DEL ANÁLISIS FORENSE DUAL:")
    if es_fraude:
        print(f"  🚨 RESULTADO: [FRAUDE DETECTADO]")
        print(f"  ⚠️ Probabilidad de Fraude: {prob_fraude*100:.2f}%")
        print("  ⚠️ Causa: La imagen no coincide con el diseño oficial de Nequi o presenta alteraciones forenses.")
    else:
        print(f"  ✅ RESULTADO: [COMPROBANTE AUTÉNTICO]")
        print(f"  🛡️ Probabilidad de Autenticidad: {(1-prob_fraude)*100:.2f}%")
        print("  🛡️ Validación: Se confirmó el diseño oficial del comprobante (QR y formato) y homogeneidad de compresión.")
    print("="*60)